# AI-Powered Financial Document Fraud Detector

> **Portfolio version of an academic group project completed at LUISS Guido Carli University** for the *Introduction to Artificial Intelligence* course.
> This is an educational prototype, not a production fraud-adjudication system and not a claim of research novelty.

This notebook implements a multi-channel fraud-risk assessment pipeline combining:
- **Multimodal AI** for document-image analysis
- **Supervised machine learning** for metadata classification
- **Multi-agent LLM assessment** using three specialised expert personas
- **Forensic rule-based checks** for near-duplicate invoices
- **Reinforcement-learning analysis** describing possible future optimisation (analysis only; no RL agent is deployed)

API credentials are loaded only from environment variables. See `.env.example` in the repository root.


In [ ]:
import os
import json
import time
import re
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import google.generativeai as genai
from PIL import Image

from dotenv import load_dotenv
load_dotenv()

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")

if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)

# Gemma 4 31B (instruction-tuned) — open-weights Google model served via
# the Gemini API. Multimodal (text + vision), 256K context, free-tier eligible.
TEXT_MODEL = "gemma-4-31b-it"
VISION_MODEL = "gemma-4-31b-it"

# Vision-channel toggle. False -> Gemma 4 31B (default, uses GEMINI_API_KEY).
# True  -> NVIDIA Nemotron Nano 12B VL via OpenRouter (requires OPENROUTER_API_KEY).
# Falls back to Gemma at runtime if Nemotron is selected but no OpenRouter key
# is configured (cell 5 prints a warning and switches transparently).
USE_NEMOTRON_VISION = True

np.random.seed(42)
sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

In [ ]:
# =============================================================================
# DATA — REAL KAGGLE DATA + ORIGINAL SYNTHETIC GENERATOR, MERGED
# -----------------------------------------------------------------------------
# Source: https://www.kaggle.com/datasets/tokelomashile2/procurement-invoice-fraud-dataset
#   300,000 invoices, 22% labelled fraud across 5 types, plus supplier /
#   department / behavioural feature tables and 45k rendered images.
#
# Strategy:
#   1. Stratified-sample 10,000 real invoices from Kaggle (keep 22% fraud).
#   2. Generate N synthetic invoices with the original pre-Kaggle recipe:
#      lognormal(4, 1.5) amounts, random weekend/rounded/dup flags, and
#      the hand-crafted `_fraud_probability` generative label model.
#      Kaggle-only columns (supplier_risk_score, blacklisted_flag,
#      split_invoice_flag, late_night_flag, amount_zscore, supplier_country,
#      fraud_type) are sampled from the Kaggle marginals so synthetic rows
#      are schema-compatible — otherwise NaN/zero fill would leak
#      "row is synthetic" into the RF.
#   3. Concat → one unified dataset → stratified 80/20 split.
# =============================================================================
import kagglehub
from sklearn.model_selection import train_test_split

_kaggle_root = kagglehub.dataset_download("tokelomashile2/procurement-invoice-fraud-dataset")
_data_dir = os.path.join(_kaggle_root, "Procument Invoice Fraud Dataset v1")

invoices_raw    = pd.read_parquet(os.path.join(_data_dir, "invoices.parquet"))
labels_raw      = pd.read_parquet(os.path.join(_data_dir, "labels.parquet"))
suppliers_raw   = pd.read_parquet(os.path.join(_data_dir, "suppliers.parquet"))
behavioural_raw = pd.read_parquet(os.path.join(_data_dir, "behavioural_features.parquet"))

merged = (invoices_raw
          .merge(labels_raw[["invoice_id", "is_fraud", "fraud_type"]],
                 on="invoice_id", how="inner")
          .merge(suppliers_raw[["supplier_id", "supplier_country",
                                "supplier_risk_score", "blacklisted_flag"]],
                 on="supplier_id", how="left")
          .merge(behavioural_raw[["invoice_id", "duplicate_invoice_flag",
                                  "split_invoice_flag", "late_night_submission_flag",
                                  "invoice_amount_zscore"]],
                 on="invoice_id", how="left"))

# Pre-sample near-duplicate pass: compute on the FULL 300k Kaggle table so
# pairs survive the stratified subsample. (If we compute post-sample the
# 10k stratified pick breaks ~96% of DUPLICATE pairs.)
from collections import defaultdict as _dd
def _kaggle_near_dup(raw, tol=5.0, win_days=7):
    flag = np.zeros(len(raw), dtype=np.int8)
    dates = pd.to_datetime(raw["invoice_date"]).values.astype("datetime64[D]")
    supplier = raw["supplier_id"].values
    amount   = raw["invoice_amount"].values
    by_s = _dd(list)
    for i in range(len(raw)):
        by_s[supplier[i]].append(i)
    for idxs in by_s.values():
        if len(idxs) < 2: continue
        idxs = sorted(idxs, key=lambda k: dates[k])
        j = 0
        for i in range(len(idxs)):
            ii = idxs[i]; di = dates[ii]; ai = amount[ii]
            while j < i and (di - dates[idxs[j]]).astype("timedelta64[D]").astype(int) > win_days:
                j += 1
            for k in range(j, i):
                ik = idxs[k]
                if abs(ai - amount[ik]) < tol:
                    flag[ii] = 1; flag[ik] = 1
                    break
    return flag

merged["near_duplicate_flag_full"] = _kaggle_near_dup(merged)

KAGGLE_SAMPLE = 10_000
merged, _ = train_test_split(
    merged, train_size=KAGGLE_SAMPLE,
    stratify=merged["is_fraud"], random_state=42,
)
merged = merged.reset_index(drop=True)

_dates = pd.to_datetime(merged["invoice_date"])
kaggle_df = pd.DataFrame({
    "document_id":               merged["invoice_id"],
    "document_type":             merged["invoice_type"].str.lower(),
    "amount":                    merged["invoice_amount"].astype(float),
    "date_created":              _dates,
    "vendor_name":               merged["supplier_id"].str.slice(0, 8),
    "is_weekend_transaction":    _dates.dt.dayofweek.isin([5, 6]).astype(int),
    "amount_rounded":            (merged["invoice_amount"].round(2) % 1 == 0).astype(int),
    "duplicate_vendor_same_day": merged["duplicate_invoice_flag"].fillna(0).astype(int),
    "supplier_country":          merged["supplier_country"].fillna("UNK"),
    "supplier_risk_score":       merged["supplier_risk_score"].fillna(0.0),
    "blacklisted_flag":          merged["blacklisted_flag"].fillna(0).astype(int),
    "split_invoice_flag":        merged["split_invoice_flag"].fillna(0).astype(int),
    "late_night_flag":           merged["late_night_submission_flag"].fillna(0).astype(int),
    "amount_zscore":             merged["invoice_amount_zscore"].fillna(0.0),
    "near_duplicate_flag":       merged["near_duplicate_flag_full"].fillna(0).astype(int),
    "fraud_type":                merged["fraud_type"],
    "is_fraud":                  merged["is_fraud"].astype(int),
    "source":                    "kaggle",
})

# -----------------------------------------------------------------------------
# Original pre-Kaggle synthetic generator, restored verbatim.
# -----------------------------------------------------------------------------
np.random.seed(42)
N_SYNTH = 2_000  # ~20% of the Kaggle sample — enough to register without dominating

_synth_raw = pd.DataFrame({
    "document_id":               [f"SYN_{i:05d}" for i in range(N_SYNTH)],
    "document_type":              np.random.choice(["receipt", "invoice", "bank_statement"], N_SYNTH),
    "amount":                     np.random.lognormal(4, 1.5, N_SYNTH).round(2),
    "date_created":               pd.date_range(start="2023-01-01", periods=N_SYNTH, freq="h"),
    "vendor_name":               [f"Vendor_{np.random.randint(1, 21)}" for _ in range(N_SYNTH)],
    "is_weekend_transaction":     np.random.choice([0, 1], N_SYNTH, p=[0.8, 0.2]),
    "amount_rounded":             np.random.choice([0, 1], N_SYNTH, p=[0.7, 0.3]),
    "duplicate_vendor_same_day":  np.random.choice([0, 1], N_SYNTH, p=[0.9, 0.1]),
})


def _fraud_probability(row) -> float:
    """Generative fraud model — features causally raise fraud likelihood."""
    p = 0.04
    if row["is_weekend_transaction"]:       p += 0.18
    if row["amount_rounded"]:                p += 0.12
    if row["duplicate_vendor_same_day"]:     p += 0.35
    if row["amount"] > 250:                  p += 0.15
    if row["document_type"] == "bank_statement": p += 0.05
    return float(min(p, 0.95))


_fraud_rng = np.random.RandomState(7)
_synth_raw["is_fraud"] = (
    _fraud_rng.rand(N_SYNTH) < _synth_raw.apply(_fraud_probability, axis=1).values
).astype(int)

# Fill the Kaggle-only columns on synthetic rows by sampling from Kaggle marginals
# — keeps the schema unified without painting a "synthetic row" fingerprint
# that the RF would learn to separate on.
_rng = np.random.RandomState(11)
synth_df = _synth_raw.copy()
synth_df["supplier_country"]    = _rng.choice(kaggle_df["supplier_country"].values,    N_SYNTH)
synth_df["supplier_risk_score"] = _rng.choice(kaggle_df["supplier_risk_score"].values, N_SYNTH)
synth_df["blacklisted_flag"]    = _rng.choice(kaggle_df["blacklisted_flag"].values,    N_SYNTH)
synth_df["split_invoice_flag"]  = _rng.choice(kaggle_df["split_invoice_flag"].values,  N_SYNTH)
synth_df["late_night_flag"]     = _rng.choice(kaggle_df["late_night_flag"].values,     N_SYNTH)
synth_df["amount_zscore"]       = _rng.choice(kaggle_df["amount_zscore"].values,       N_SYNTH)
synth_df["fraud_type"]          = np.where(synth_df["is_fraud"] == 1, "SYNTHETIC", "NONE")
synth_df["source"]              = "synthetic"
synth_df["near_duplicate_flag"] = 0  # overwritten by cell 9 after concat

# Unified single dataset — real + synthetic, in column order.
df = pd.concat([kaggle_df, synth_df[kaggle_df.columns]], ignore_index=True)
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

df.to_csv("data.csv", index=False)
# 80-20 stratified split on the combined frame.
train_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df["is_fraud"], random_state=42,
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Combined dataset: {len(df):,} rows "
      f"({len(kaggle_df):,} Kaggle + {len(synth_df):,} synthetic)")
print(f"  Kaggle   fraud rate: {kaggle_df['is_fraud'].mean():.1%}")
print(f"  Synth    fraud rate: {synth_df['is_fraud'].mean():.1%}")
print(f"  Combined fraud rate: {df['is_fraud'].mean():.1%}")
print(f"  Train: {len(train_df):,} rows  ({train_df['is_fraud'].mean():.1%} fraud)")
print(f"  Test : {len(test_df):,} rows  ({test_df['is_fraud'].mean():.1%} fraud)")
print("Fraud types in sample:")
print(df.loc[df["is_fraud"] == 1, "fraud_type"].value_counts().to_string())
df.head()

In [ ]:
print("Dataset Shape:", df.shape)
print("\nStatistical Summary:")
print(df.describe())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df['document_type'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Document Types')
axes[0].tick_params(axis='x', rotation=30)

axes[1].hist(df['amount'], bins=25, color='darkorange', alpha=0.8)
axes[1].set_title('Amount Distribution')
axes[1].set_xlabel('Amount')

df['is_fraud'].value_counts().plot(kind='bar', ax=axes[2], color=['seagreen', 'crimson'])
axes[2].set_title('Fraud Distribution')
axes[2].set_xticklabels(['Not Fraud', 'Fraud'], rotation=0)

plt.tight_layout()
plt.show()

## 1. Image Analysis (Multimodal AI)

`load_and_analyze_image` uses Gemini Vision to look for visual fraud indicators: inconsistent fonts, misaligned text, altered numbers, suspicious layouts. Returns a JSON-parsed structured result.

In [ ]:
IMAGE_PROMPT = """You are a forensic document examiner. Analyze the attached financial document image
(receipt, invoice, or bank statement) for visual fraud indicators.

Check for: inconsistent fonts/typography, misaligned text, pixelation around numbers,
altered totals, mismatched logos, suspicious layouts, copy-paste artifacts, unusual whitespace.

Respond ONLY with valid JSON in this exact schema:
{
  "document_type": "receipt|invoice|bank_statement|unknown",
  "fraud_indicators": ["..."],
  "visual_confidence": 0.0,
  "fraud_risk_score": 0,
  "summary": "one short sentence"
}
where visual_confidence is 0.0-1.0 and fraud_risk_score is 0-100."""


def _extract_json(text: str) -> dict:
    """Pull the first JSON object out of an LLM response, tolerating code fences."""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return {"_raw": text, "_parse_error": True}
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return {"_raw": text, "_parse_error": True}


def load_and_analyze_image(image_path: str) -> dict:
    """Analyze a financial document image for fraud indicators via Gemini Vision."""
    if not os.path.exists(image_path):
        return {"error": f"Image not found: {image_path}", "fraud_risk_score": 0}

    image = Image.open(image_path)
    model = genai.GenerativeModel(VISION_MODEL)
    response = model.generate_content([IMAGE_PROMPT, image])
    result = _extract_json(response.text)
    result["image_path"] = image_path
    return result


print("Image analyzer ready. Call load_and_analyze_image('path/to/doc.png') when images are available.")


# =============================================================================
# Dual-vision: NVIDIA Nemotron Nano VL via OpenRouter (course brief BONUS)
# -----------------------------------------------------------------------------
# Adds a second, independent vision model alongside Gemini. Both models analyze
# the same image with the same forensic-examiner prompt, and the results are
# aggregated per the **Parallelization → Sectioning** workflow pattern
# (Lecture 09 slide 28). Two independent models = error decorrelation; same
# logic that makes Random Forest and the multi-agent committee work.
# =============================================================================
import base64
from openai import OpenAI

NEMOTRON_VISION_MODEL = "nvidia/nemotron-nano-12b-v2-vl:free"

_openrouter = (
    OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)
    if OPENROUTER_API_KEY else None
)


def _image_to_data_url(image_path: str) -> str:
    """Encode a local image as a base64 data URL for OpenRouter's vision endpoint."""
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("ascii")
    ext = os.path.splitext(image_path)[1].lower().lstrip(".")
    mime = {"png": "image/png", "jpg": "image/jpeg", "jpeg": "image/jpeg"}.get(ext, "image/png")
    return f"data:{mime};base64,{b64}"


def load_and_analyze_image_nemotron(image_path: str) -> dict:
    """Same forensic-examiner prompt, but routed to NVIDIA Nemotron Nano VL.

    Returns the same JSON schema as `load_and_analyze_image` so the dual aggregator
    can treat both models interchangeably.
    """
    if _openrouter is None:
        return {"error": "OPENROUTER_API_KEY not set", "fraud_risk_score": 0,
                "image_path": image_path, "model": NEMOTRON_VISION_MODEL}
    if not os.path.exists(image_path):
        return {"error": f"Image not found: {image_path}", "fraud_risk_score": 0,
                "image_path": image_path, "model": NEMOTRON_VISION_MODEL}

    response = _openrouter.chat.completions.create(
        model=NEMOTRON_VISION_MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": IMAGE_PROMPT},
                {"type": "image_url", "image_url": {"url": _image_to_data_url(image_path)}},
            ],
        }],
    )
    result = _extract_json(response.choices[0].message.content or "")
    result["image_path"] = image_path
    result["model"] = NEMOTRON_VISION_MODEL
    return result


def load_and_analyze_image_dual(image_path: str, sleep_s: float = 1.0) -> dict:
    """Run Gemini and Nemotron on the same image, return one aggregated result.

    Aggregation:
        - score: mean when models agree (spread < 20), else 60% mean + 40% max
          — same disagreement-weighted blend used in the multi-agent assessor,
          biased toward the higher score because false negatives are costlier
          than false positives in fraud review.
        - fraud_indicators: deduplicated union of both lists.
        - visual_confidence: mean.
        - per_model: full raw outputs preserved for the report.

    Graceful fallback: if either model errors out, return the surviving one
    unchanged (so the existing pipeline keeps running with one less channel).
    """
    gemini_result = load_and_analyze_image(image_path)
    if sleep_s > 0:
        time.sleep(sleep_s)
    nemotron_result = load_and_analyze_image_nemotron(image_path)

    available = [(name, r) for name, r in
                 [("gemini", gemini_result), ("nemotron", nemotron_result)]
                 if "error" not in r and not r.get("_parse_error")]

    if not available:
        return {"error": "Both vision models failed", "fraud_risk_score": 0,
                "image_path": image_path,
                "per_model": {"gemini": gemini_result, "nemotron": nemotron_result}}

    scores = [float(r.get("fraud_risk_score", 0) or 0) for _, r in available]
    confs  = [float(r.get("visual_confidence", 0) or 0) for _, r in available]

    indicators: list[str] = []
    for _, r in available:
        for ind in (r.get("fraud_indicators") or []):
            if ind not in indicators:
                indicators.append(ind)

    spread = float(max(scores) - min(scores)) if len(scores) > 1 else 0.0
    consensus = "consensus" if spread < 20 else "disagree"
    if consensus == "consensus" or len(scores) == 1:
        final_score = float(np.mean(scores))
    else:
        final_score = 0.6 * float(np.mean(scores)) + 0.4 * float(max(scores))

    summaries = [r.get("summary", "") for _, r in available if r.get("summary")]
    return {
        "image_path": image_path,
        "fraud_risk_score": final_score,
        "visual_confidence": float(np.mean(confs)),
        "fraud_indicators": indicators,
        "consensus": consensus,
        "score_spread": spread,
        "summary": " | ".join(summaries),
        "per_model": {name: r for name, r in [("gemini", gemini_result),
                                              ("nemotron", nemotron_result)]},
    }


def vision_analyze(image_path: str) -> dict:
    """Single entry point that respects the cell-1 USE_NEMOTRON_VISION toggle.

    USE_NEMOTRON_VISION=False  -> Gemma 4 31B (load_and_analyze_image)
    USE_NEMOTRON_VISION=True   -> Nemotron Nano 12B VL (load_and_analyze_image_nemotron)

    If Nemotron is selected but OPENROUTER_API_KEY is missing, transparently
    falls back to Gemma so the pipeline still runs.
    """
    if USE_NEMOTRON_VISION:
        if _openrouter is None:
            print("  [warn] USE_NEMOTRON_VISION=True but OPENROUTER_API_KEY not set; "
                  "falling back to Gemma.")
            return load_and_analyze_image(image_path)
        return load_and_analyze_image_nemotron(image_path)
    return load_and_analyze_image(image_path)


_active = "Nemotron" if (USE_NEMOTRON_VISION and _openrouter is not None) else "Gemma"
print(f"Vision channel ready. Active model: {_active}.")
print(f"  Toggle in cell 1: USE_NEMOTRON_VISION={USE_NEMOTRON_VISION}, "
      f"OpenRouter configured: {_openrouter is not None}")
print(f"  Dual-vision (Gemma + Nemotron together) is still available as "
      f"`load_and_analyze_image_dual(path)`.")

## 2. Metadata Analysis — ML Approach

In [ ]:
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score
from sklearn.calibration import CalibratedClassifierCV


def analyze_metadata_ml(train_df: pd.DataFrame, test_df: pd.DataFrame,
                         full_df: pd.DataFrame | None = None) -> dict:
    """Train a calibrated Random Forest on the 80% split, evaluate on 20% held-out.

    Data regime change vs. the synthetic version:
    - ~8,000 training rows and ~2,000 test rows (22% fraud base rate) —
      ample data for a single honest 80-20 split per the assignment spec.
      With ~440 test-set frauds, reported recall/precision are stable
      (standard error ≈ 2 pp) — CV is no longer needed to fight small-N noise.

    Pipeline-applied theory (Lecture 05):
    - ``max_features='sqrt'`` — explicit √p decorrelation (slide 17).
    - ``n_estimators=400`` — more trees never hurts (slide 22).
    - ``class_weight='balanced'`` — 22% base rate is mild but still biased.
    - ``CalibratedClassifierCV`` (isotonic, 5-fold over the train set) —
      the composite scorer in cell 13 runs RF probability through a sigmoid;
      miscalibrated input would distort the amplified score.

    Feature set — union of the synthetic notebook's features and the real
    dataset's supplier / behavioural enrichments (13 features total).
    """
    def featurize(frame: pd.DataFrame,
                   vendor_counts: pd.Series | None = None
                   ) -> tuple[pd.DataFrame, pd.Series]:
        f = frame.copy()
        f["doc_type_encoded"] = f["document_type"].astype("category").cat.codes
        f["amount_log"]       = np.log1p(f["amount"])
        f["amount_gt_250"]    = (f["amount"] > 250).astype(int)
        f["day_of_week"]      = pd.to_datetime(f["date_created"]).dt.dayofweek
        if vendor_counts is None:
            vendor_counts = f["vendor_name"].value_counts()
        f["vendor_frequency"] = f["vendor_name"].map(vendor_counts).fillna(1).astype(int)
        return f, vendor_counts

    feature_cols = [
        "doc_type_encoded", "vendor_frequency", "amount_log", "amount_gt_250",
        "day_of_week", "is_weekend_transaction", "amount_rounded",
        "duplicate_vendor_same_day", "split_invoice_flag", "late_night_flag",
        "supplier_risk_score", "blacklisted_flag", "amount_zscore",
    ]

    train_feat, vendor_counts = featurize(train_df)
    test_feat, _              = featurize(test_df, vendor_counts)
    X_train, y_train = train_feat[feature_cols], train_feat["is_fraud"]
    X_test,  y_test  = test_feat[feature_cols],  test_feat["is_fraud"]

    base_rf = RandomForestClassifier(
        n_estimators=400,
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    model = CalibratedClassifierCV(base_rf, method="isotonic", cv=5)
    model.fit(X_train, y_train)

    # Threshold tuned on training F1 with a 0.15 floor to avoid flag-everything.
    train_proba = model.predict_proba(X_train)[:, 1]
    candidate_thresholds = np.linspace(0.15, 0.9, 76)
    f1_train = [f1_score(y_train, (train_proba >= t).astype(int), zero_division=0)
                for t in candidate_thresholds]
    fraud_threshold = float(candidate_thresholds[int(np.argmax(f1_train))])

    # Honest held-out evaluation on the 20% test split.
    test_proba = model.predict_proba(X_test)[:, 1]
    test_pred  = (test_proba >= fraud_threshold).astype(int)
    test_auc   = roc_auc_score(y_test, test_proba)

    importance_matrix = np.array([
        cal.estimator.feature_importances_ for cal in model.calibrated_classifiers_
    ])
    importance = pd.Series(importance_matrix.mean(axis=0),
                           index=feature_cols).sort_values(ascending=False)

    # Score every row in the original `df` so cell 13's integrated scorer can
    # look up a fraud probability per document_id.
    if full_df is None:
        full_df = pd.concat([train_df, test_df], ignore_index=True)
    full_feat, _ = featurize(full_df, vendor_counts)
    full_proba = model.predict_proba(full_feat[feature_cols])[:, 1]

    return {
        "model": model,
        "feature_cols": feature_cols,
        "vendor_counts": vendor_counts,
        "fraud_threshold": fraud_threshold,
        # Honest 80/20 test metrics — headline numbers for the assignment.
        "test_roc_auc":   test_auc,
        "test_recall":    recall_score(y_test, test_pred, zero_division=0),
        "test_precision": precision_score(y_test, test_pred, zero_division=0),
        "test_f1":        f1_score(y_test, test_pred, zero_division=0),
        "test_accuracy":  accuracy_score(y_test, test_pred),
        "classification_report": classification_report(y_test, test_pred, zero_division=0),
        "confusion_matrix":      confusion_matrix(y_test, test_pred),
        "feature_importance":    importance,
        "test_probabilities":    test_proba,
        "test_y_true":           y_test.values,
        "test_document_ids":     test_df["document_id"].values,
        "full_probabilities":    full_proba,
        # Back-compat aliases: downstream cells were written for the CV-OOF
        # version of this function; these keys preserve that interface.
        "cv_roc_auc":       test_auc,
        "oof_recall":       recall_score(y_test, test_pred, zero_division=0),
        "oof_precision":    precision_score(y_test, test_pred, zero_division=0),
        "oof_f1":           f1_score(y_test, test_pred, zero_division=0),
        "oof_accuracy":     accuracy_score(y_test, test_pred),
        "oof_probabilities": test_proba,
    }


ml_results = analyze_metadata_ml(train_df, test_df, full_df=df)
print("Random Forest — trained on 80% split, evaluated on 20% held-out test:")
print(f"  Test ROC-AUC:   {ml_results['test_roc_auc']:.3f}")
print(f"  Test Recall:    {ml_results['test_recall']:.2%}   (primary KPI — missed frauds are costly)")
print(f"  Test Precision: {ml_results['test_precision']:.2%}")
print(f"  Test F1:        {ml_results['test_f1']:.3f}")
print(f"  Test Accuracy:  {ml_results['test_accuracy']:.2%}")
print(f"  Threshold:      {ml_results['fraud_threshold']:.2f}   (data-driven, tuned on train F1)")
print("\nFeature importance (avg across calibration folds):")
print(ml_results["feature_importance"])
print("\nClassification report (held-out 20% test set):")
print(ml_results["classification_report"])


## 2.5  Fuzzy / Near-Duplicate Payment Detection (Forensic-Audit Method)

Classic SAP GRC / Oracle audit rule: flag invoice pairs where the same supplier submits two invoices whose amounts differ by less than a tolerance (\$5 here) within a 7-day window. Catches duplicate-payment fraud where the fraudster tweaks the amount or the date slightly to evade exact-match deduplication — a blind spot of the Kaggle `duplicate_invoice_flag`, which flags exact same-day duplicates only.

The check runs directly on `df` and adds a `near_duplicate_flag` column so the downstream RF and integration layer can consume it.

In [ ]:
# The Kaggle rows already carry `near_duplicate_flag` computed on the full
# 300k invoice table (cell 2). Here we only need to run the same rule on the
# synthetic slice — the Kaggle slice's flag is already authoritative.
from collections import defaultdict

AMOUNT_TOLERANCE = 5.0    # USD; same-supplier amounts within this count as near-match
DATE_WINDOW_DAYS = 7


def compute_near_duplicate_flag(d: pd.DataFrame) -> pd.Series:
    """Flag rows belonging to a near-duplicate invoice pair.

    Canonical forensic-audit rule (SAP GRC / Oracle invoice-matching audit):
    two invoices from the same supplier, amounts within $5, dates within 7 days.

    O(n log n): partition by supplier, sort each partition by date, sweep with
    a two-pointer 7-day window. Positional indexing so it works on any slice.
    """
    n = len(d)
    flag = np.zeros(n, dtype=int)
    vendor = d["vendor_name"].to_numpy()
    date   = d["date_created"].to_numpy()
    amount = d["amount"].to_numpy()
    by_vendor = defaultdict(list)
    for pos in range(n):
        by_vendor[vendor[pos]].append(pos)
    for positions in by_vendor.values():
        if len(positions) < 2:
            continue
        positions.sort(key=lambda p: date[p])
        j = 0
        for i in range(len(positions)):
            pi = positions[i]
            while j < i and (pd.Timestamp(date[pi]) - pd.Timestamp(date[positions[j]])).days > DATE_WINDOW_DAYS:
                j += 1
            for k in range(j, i):
                pk = positions[k]
                if abs(amount[pi] - amount[pk]) < AMOUNT_TOLERANCE:
                    flag[pi] = 1
                    flag[pk] = 1
                    break
    return pd.Series(flag, index=d.index)


# Only compute on the synthetic slice — Kaggle rows already have the full-table flag.
synth_mask = df["source"] == "synthetic"
df.loc[synth_mask, "near_duplicate_flag"] = (
    compute_near_duplicate_flag(df.loc[synth_mask]).values
)
df["near_duplicate_flag"] = df["near_duplicate_flag"].astype(int)

n_flagged = int(df["near_duplicate_flag"].sum())
print(f"Near-duplicate rows flagged: {n_flagged:,}/{len(df):,} ({n_flagged/len(df):.1%})")

existing = df["duplicate_vendor_same_day"].astype(bool)
nd       = df["near_duplicate_flag"].astype(bool)
print(f"  Overlap with duplicate_vendor_same_day: {int((nd & existing).sum()):,} rows")
print(f"  Additional signal beyond it           : {int((nd & ~existing).sum()):,} rows")

print("\nRecall by fraud type (how many of each class we catch):")
for ftype in ["DUPLICATE", "SPLIT", "GHOST_SUPPLIER", "INFLATED",
              "DOC_TAMPER", "SYNTHETIC"]:
    sub = df[df["fraud_type"] == ftype]
    if len(sub):
        print(f"  {ftype:14s}  n={len(sub):>5}  "
              f"recall={sub['near_duplicate_flag'].mean():>6.1%}")
legit = df[df["is_fraud"] == 0]
print(f"  {'Legitimate':14s}  n={len(legit):>5}  "
      f"FPR   ={legit['near_duplicate_flag'].mean():>6.1%}")

print("\nPer-source breakdown (the two halves of the merged dataset behave very "
      "differently — synthetic has only 20 vendors so near-duplicates are structural "
      "noise there):")
for src in ["kaggle", "synthetic"]:
    leg_src = df[(df["is_fraud"] == 0) & (df["source"] == src)]
    frd_src = df[(df["is_fraud"] == 1) & (df["source"] == src)]
    print(f"  {src:10s}  legit n={len(leg_src):>5}  FPR={leg_src['near_duplicate_flag'].mean():>6.1%} "
          f"| fraud n={len(frd_src):>5}  recall={frd_src['near_duplicate_flag'].mean():>6.1%}")

from sklearn.metrics import roc_auc_score
kag = df[df["source"] == "kaggle"]
print(f"\nStandalone AUC (Kaggle rows only):    {roc_auc_score(kag['is_fraud'], kag['near_duplicate_flag']):.3f}")
print(f"Standalone AUC (full combined df):    {roc_auc_score(df['is_fraud'], df['near_duplicate_flag']):.3f}")

flagged_fraud_rate = df.loc[df["near_duplicate_flag"] == 1, "is_fraud"].mean()
base_rate          = df["is_fraud"].mean()
print(f"\nPrecision: flagged rows are {flagged_fraud_rate/base_rate:.2f}x more likely to "
      f"be fraud than the base rate ({flagged_fraud_rate:.1%} vs {base_rate:.1%}).")

## 3. Metadata Analysis — LLM Approach

In [ ]:
LLM_CLASSIFIER_PROMPT = """You are a fraud detection classifier. Given a batch of financial document records,
predict whether each is fraudulent (1) or legitimate (0).

Signals that raise suspicion: weekend transactions, rounded amounts, duplicate vendor same day,
unusually large amounts, or multiple weak signals combined.

Records:
{records}

Respond with ONLY a raw JSON array (no markdown, no prose, no code fences), one object per
record in the same order, using double quotes:
[{{"document_id": "DOC_0000", "pred": 0, "reason": "short"}}, ...]"""


def _row_to_text(row: pd.Series) -> str:
    return (
        f"{row['document_id']} | type={row['document_type']} amount={row['amount']:.2f} "
        f"weekend={row['is_weekend_transaction']} rounded={row['amount_rounded']} "
        f"dup_same_day={row['duplicate_vendor_same_day']} vendor={row['vendor_name']}"
    )


def _parse_llm_json_array(text: str) -> list:
    """Best-effort JSON-array extraction from an LLM response.

    Handles code fences, stray prose, and per-object recovery when the
    outer array doesn't round-trip (common with open-weights models like
    Gemma that aren't tuned for strict JSON).
    """
    cleaned = re.sub(r"```(?:json)?", "", text).replace("```", "").strip()
    start, end = cleaned.find("["), cleaned.rfind("]")
    if start == -1 or end == -1:
        return []
    blob = cleaned[start:end + 1]
    try:
        return json.loads(blob)
    except json.JSONDecodeError:
        items = []
        for m in re.finditer(r"\{[^{}]*\}", blob):
            try:
                items.append(json.loads(m.group(0)))
            except json.JSONDecodeError:
                continue
        return items


def analyze_metadata_llm(df: pd.DataFrame, batch_size: int = 20, sample: int | None = 40) -> dict:
    """Classify fraud with an LLM. `sample` caps rows to keep API calls cheap."""
    working = df.sample(n=sample, random_state=42) if sample and sample < len(df) else df
    model = genai.GenerativeModel(TEXT_MODEL)

    predictions: dict[str, int] = {}
    reasons: dict[str, str] = {}

    for start in range(0, len(working), batch_size):
        batch = working.iloc[start:start + batch_size]
        records = "\n".join(_row_to_text(r) for _, r in batch.iterrows())
        prompt = LLM_CLASSIFIER_PROMPT.format(records=records)
        try:
            response = model.generate_content(prompt)
            items = _parse_llm_json_array(response.text)
            if not items:
                print(f"  batch {start}: no items parsed. Raw response preview:")
                print(f"    {response.text[:300]!r}")
            for item in items:
                doc_id = item.get("document_id")
                if not doc_id:
                    continue
                predictions[doc_id] = int(item.get("pred", 0))
                reasons[doc_id] = item.get("reason", "")
        except Exception as exc:
            print(f"  batch {start} failed: {exc}")
        time.sleep(1)  # gentle rate limiting for free tier

    aligned = working[working['document_id'].isin(predictions)].copy()
    aligned['llm_pred'] = aligned['document_id'].map(predictions)
    aligned['llm_reason'] = aligned['document_id'].map(reasons)

    accuracy = accuracy_score(aligned['is_fraud'], aligned['llm_pred']) if len(aligned) else 0.0

    return {
        'predictions': aligned[['document_id', 'is_fraud', 'llm_pred', 'llm_reason']],
        'accuracy': accuracy,
        'n_classified': len(aligned),
    }


# Uncomment to run (uses API quota):
llm_results = analyze_metadata_llm(df, sample=30)
print(f"LLM classifier accuracy: {llm_results['accuracy']:.2%} on {llm_results['n_classified']} docs")
llm_results['predictions'].head()
#print("LLM classifier defined. Uncomment the call above to run against the Gemini API.")


## 4. Multi-Agent Risk Assessment

Three expert personas evaluate the same document through distinct system prompts.

In [ ]:
EXPERT_PROMPTS = {
    'forensic_accountant': (
        "You are a senior forensic accountant with 20 years of experience investigating financial fraud. "
        "Focus on: financial inconsistencies, unusual amounts, accounting red flags, vendor patterns, "
        "transaction timing. Be precise and cite which fields drove your conclusion."
    ),
    'compliance_officer': (
        "You are a compliance officer specializing in AML and KYC. "
        "Focus on: regulatory red flags, structuring, duplicate transactions, suspicious vendor relationships, "
        "documentation gaps. Frame your reasoning in terms of policy violations."
    ),
    'risk_analyst': (
        "You are a quantitative risk analyst. "
        "Focus on: statistical anomalies, outlier detection, base-rate reasoning, combined-signal risk. "
        "Reason about how many weak signals stack, not just any single one."
    ),
}

AGENT_TEMPLATE = """{system}

Document under review:
{doc}

Respond ONLY with valid JSON:
{{
  "risk_score": <int 1-10>,
  "verdict": "low|medium|high",
  "key_concerns": ["..."],
  "rationale": "2-3 sentences"
}}"""


def _doc_to_text(doc: dict | pd.Series) -> str:
    d = dict(doc)
    return (
        f"ID: {d.get('document_id')}\n"
        f"Type: {d.get('document_type')}\n"
        f"Amount: {d.get('amount')}\n"
        f"Date: {d.get('date_created')}\n"
        f"Vendor: {d.get('vendor_name')}\n"
        f"Weekend transaction: {d.get('is_weekend_transaction')}\n"
        f"Rounded amount: {d.get('amount_rounded')}\n"
        f"Duplicate vendor same day: {d.get('duplicate_vendor_same_day')}"
    )


def get_expert_opinion(document_data, expert_type: str) -> dict:
    if expert_type not in EXPERT_PROMPTS:
        raise ValueError(f"Unknown expert: {expert_type}")
    prompt = AGENT_TEMPLATE.format(
        system=EXPERT_PROMPTS[expert_type],
        doc=_doc_to_text(document_data),
    )
    model = genai.GenerativeModel(TEXT_MODEL)
    response = model.generate_content(prompt)
    parsed = _extract_json(response.text)
    parsed['expert_type'] = expert_type
    return parsed


def multi_agent_assessment(document_data, sleep_s: float = 7.0) -> dict:
    """Run every expert sequentially, aggregate scores, flag disagreement.

    sleep_s spaces calls to stay under the Gemini Flash free-tier 10 RPM cap;
    callers running long batches should keep the default."""
    opinions = []
    for expert in EXPERT_PROMPTS:
        try:
            opinions.append(get_expert_opinion(document_data, expert))
        except Exception as exc:
            opinions.append({'expert_type': expert, 'error': str(exc), 'risk_score': 0})
        time.sleep(sleep_s)

    scores = [o.get('risk_score', 0) or 0 for o in opinions]
    avg_score = float(np.mean(scores)) if scores else 0.0
    max_score = float(np.max(scores)) if scores else 0.0
    disagreement = float(np.std(scores)) if scores else 0.0
    consensus = "agree" if disagreement < 1.5 else "disagree"

    # Condorcet-style aggregation (Lecture 05 slide 15 + Lecture 09 slide 28).
    # A single expert flagging 8+ is a strong signal even without consensus —
    # fraud review is asymmetric, we'd rather investigate one false lead than
    # miss a real one. Blend mean (for stable base rate) with max (to preserve
    # strong minority-expert signals). Weight shifts toward max when experts
    # disagree — disagreement itself is information, not noise.
    disagreement_weight = min(disagreement / 3.0, 0.5)  # capped at 50% max-weight
    aggregated_score = (1 - disagreement_weight) * avg_score + disagreement_weight * max_score

    return {
        'opinions': opinions,
        'avg_risk_score': aggregated_score,  # feeds the integrated scorer
        'mean_risk_score': avg_score,
        'max_risk_score': max_score,
        'score_disagreement': disagreement,
        'consensus': consensus,
    }


sample_doc = df.iloc[0].to_dict()
print("Example document:", sample_doc['document_id'])
# Uncomment to run live (consumes API quota):
assessment = multi_agent_assessment(sample_doc)
print(json.dumps(assessment, indent=2, default=str))

## 5. Integration + Risk Report

In [ ]:
import math

WEIGHTS = {'image': 0.30, 'metadata': 0.40, 'agents': 0.30}
REVIEW_THRESHOLD = 35   # was 40 — lowered so REVIEW catches mid-confidence frauds
REJECT_THRESHOLD = 65   # was 70 — high-confidence flags shouldn't need a perfect score


def _amplify_metadata(prob: float) -> float:
    """Map RF fraud probability (0..1) to a 0..100 score with a steeper curve
    in the decision-relevant region. Sigmoid centered at 0.4, gain 8:
        prob 0.20 -> ~14   prob 0.40 -> 50   prob 0.60 -> ~83   prob 0.80 -> ~96
    Without amplification, a fraud doc with prob 0.5 only contributes 50*0.40=20
    to composite — never enough to trip REVIEW alone."""
    return 100.0 / (1.0 + math.exp(-8.0 * (prob - 0.4)))


def integrate_all_analyses(document_id, image_analysis, metadata_prediction, agent_assessment):
    """Combine image + metadata + multi-agent into a 0-100 composite score."""
    image_score = float(image_analysis.get('fraud_risk_score', 0)) if image_analysis else 0.0
    raw_meta_prob = float(metadata_prediction.get('fraud_probability', 0)) if metadata_prediction else 0.0
    metadata_score = _amplify_metadata(raw_meta_prob) if metadata_prediction else 0.0
    agent_score_10 = float(agent_assessment.get('avg_risk_score', 0)) if agent_assessment else 0.0
    agent_score = agent_score_10 * 10

    active = {}
    if image_analysis is not None and 'error' not in image_analysis:
        active['image'] = image_score
    if metadata_prediction is not None:
        active['metadata'] = metadata_score
    if agent_assessment is not None:
        active['agents'] = agent_score
    total_weight = sum(WEIGHTS[k] for k in active) or 1.0
    composite = sum(active[k] * WEIGHTS[k] for k in active) / total_weight

    if composite >= REJECT_THRESHOLD:
        recommendation = 'REJECT'
    elif composite >= REVIEW_THRESHOLD:
        recommendation = 'REVIEW'
    else:
        recommendation = 'APPROVE'

    risk_factors = []
    if image_analysis and image_analysis.get('fraud_indicators'):
        risk_factors.extend(f"image: {i}" for i in image_analysis['fraud_indicators'])
    if metadata_prediction and raw_meta_prob > 0.4:
        risk_factors.append(f"metadata: RF fraud probability {raw_meta_prob:.2f}")
    if agent_assessment:
        for op in agent_assessment.get('opinions', []):
            for c in op.get('key_concerns', []) or []:
                risk_factors.append(f"{op.get('expert_type')}: {c}")

    return {
        'document_id': document_id,
        'composite_risk_score': round(composite, 1),
        'recommendation': recommendation,
        'component_scores': {'image': image_score, 'metadata': metadata_score, 'agents': agent_score},
        'raw_metadata_probability': round(raw_meta_prob, 3),
        'risk_factors': risk_factors,
        'agent_consensus': agent_assessment.get('consensus') if agent_assessment else None,
    }


def generate_risk_report(document_id, integrated_assessment) -> str:
    a = integrated_assessment
    lines = [
        f"=== FRAUD RISK REPORT: {document_id} ===",
        f"Composite risk score: {a['composite_risk_score']}/100",
        f"Recommendation: {a['recommendation']}",
        f"Agent consensus: {a['agent_consensus']}",
        "Component scores:",
    ]
    for k, v in a['component_scores'].items():
        lines.append(f"  - {k}: {v:.1f}")
    lines.append("Risk factors:")
    for f in a['risk_factors'][:10]:
        lines.append(f"  - {f}")
    if a['recommendation'] == 'REJECT':
        lines.append("Next step: escalate to fraud investigations team.")
    elif a['recommendation'] == 'REVIEW':
        lines.append("Next step: manual review by compliance analyst.")
    else:
        lines.append("Next step: auto-approve, sample for random audit.")
    return "\n".join(lines)


def process_document_batch(df: pd.DataFrame, ml_results: dict, image_folder_path: str | None = None,
                           run_agents_for_top_k: int = 3) -> pd.DataFrame:
    """Batch-process: metadata for all, agents only on the top-k highest risk (API cost control)."""
    results = []
    probs = ml_results['full_probabilities']
    df_scored = df.copy()
    df_scored['fraud_probability'] = probs
    top_k_ids = set(df_scored.nlargest(run_agents_for_top_k, 'fraud_probability')['document_id'])

    for _, row in df_scored.iterrows():
        doc_id = row['document_id']
        metadata_pred = {'fraud_probability': float(row['fraud_probability'])}

        image_analysis = None
        if image_folder_path:
            for ext in ('.png', '.jpg', '.jpeg'):
                path = os.path.join(image_folder_path, doc_id + ext)
                if os.path.exists(path):
                    image_analysis = load_and_analyze_image(path)
                    break

        agent_assessment = None
        if doc_id in top_k_ids:
            try:
                agent_assessment = multi_agent_assessment(row.to_dict())
            except Exception as exc:
                agent_assessment = {'error': str(exc), 'avg_risk_score': 0, 'opinions': [], 'consensus': 'error'}

        integrated = integrate_all_analyses(doc_id, image_analysis, metadata_pred, agent_assessment)
        integrated['actual_is_fraud'] = int(row['is_fraud'])
        results.append(integrated)

    return pd.DataFrame(results)


# Demo without agents / images — pure metadata integration for all docs
demo_batch = df.copy()
demo_batch['fraud_probability'] = ml_results['full_probabilities']
demo_rows = []
for _, row in demo_batch.iterrows():
    integrated = integrate_all_analyses(
        row['document_id'],
        image_analysis=None,
        metadata_prediction={'fraud_probability': float(row['fraud_probability'])},
        agent_assessment=None,
    )
    integrated['actual_is_fraud'] = int(row['is_fraud'])
    demo_rows.append(integrated)
batch_results = pd.DataFrame(demo_rows)

# How well do the recommendations align with ground truth?
flagged = batch_results['recommendation'].isin(['REVIEW', 'REJECT'])
true_fraud = batch_results['actual_is_fraud'] == 1
caught = (flagged & true_fraud).sum()
total_fraud = true_fraud.sum()
print(f"Pipeline catches {caught}/{total_fraud} actual frauds ({caught/total_fraud:.1%} recall) "
      f"by flagging {flagged.sum()}/{len(batch_results)} documents.")
print("\nTop 5 highest-risk documents (metadata only):")
print(batch_results.nlargest(5, 'composite_risk_score')[
    ['document_id', 'composite_risk_score', 'raw_metadata_probability', 'recommendation', 'actual_is_fraud']
])

## 6. Risk Dashboard

In [ ]:
def create_risk_dashboard(batch_results: pd.DataFrame, ml_results: dict):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].hist(batch_results['composite_risk_score'], bins=20, color='steelblue', alpha=0.85)
    axes[0, 0].axvline(REVIEW_THRESHOLD, color='orange', ls='--', label=f'REVIEW ≥ {REVIEW_THRESHOLD}')
    axes[0, 0].axvline(REJECT_THRESHOLD, color='red', ls='--', label=f'REJECT ≥ {REJECT_THRESHOLD}')
    axes[0, 0].set_title('Composite Risk Score Distribution')
    axes[0, 0].set_xlabel('Risk score')
    axes[0, 0].legend()

    from sklearn.metrics import confusion_matrix as _cm
    # Confusion matrix on the held-out 20% test split (honest evaluation).
    y_true = ml_results['test_y_true']
    test_pred = (ml_results['test_probabilities'] >= ml_results['fraud_threshold']).astype(int)
    cm = _cm(y_true, test_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 1],
                xticklabels=['Not fraud', 'Fraud'], yticklabels=['Not fraud', 'Fraud'])
    axes[0, 1].set_title(
        f"RF Test Confusion Matrix — acc {ml_results['test_accuracy']:.0%}, "
        f"recall {ml_results['test_recall']:.0%}, AUC {ml_results['test_roc_auc']:.2f}"
    )
    axes[0, 1].set_xlabel('Predicted')
    axes[0, 1].set_ylabel('Actual')

    ml_results['feature_importance'].plot(kind='barh', ax=axes[1, 0], color='seagreen')
    axes[1, 0].set_title('Random Forest Feature Importance')
    axes[1, 0].invert_yaxis()

    rec_counts = batch_results['recommendation'].value_counts()
    colors = {'APPROVE': 'seagreen', 'REVIEW': 'orange', 'REJECT': 'crimson'}
    axes[1, 1].bar(rec_counts.index, rec_counts.values, color=[colors.get(r, 'grey') for r in rec_counts.index])
    axes[1, 1].set_title('Recommendation Distribution')
    axes[1, 1].set_ylabel('Document count')

    plt.tight_layout()
    plt.show()


create_risk_dashboard(batch_results, ml_results)


In [ ]:
# ===================================================================
# 6.5  REAL-DOCUMENT MULTIMODAL PIPELINE
# -------------------------------------------------------------------
# Run the full image + metadata + multi-agent flow on 12 real invoice
# images sampled from the Kaggle procurement-invoice-fraud dataset.
#
# Sampling: 6 fraud + 6 legit drawn from the intersection of our 10K
# working sample and the 45K-image subset (images_metadata.parquet),
# so every sample already carries the full 13-feature enrichment that
# the calibrated RF was trained on — no fabricated defaults.
# ===================================================================
_img_meta = pd.read_parquet(os.path.join(_data_dir, "images_metadata.parquet"))
IMAGES_DIR = os.path.join(_data_dir, "images")

# Only consider invoices we already featurised in `df`.
_with_images = df.merge(
    _img_meta[["invoice_id"]].rename(columns={"invoice_id": "document_id"}),
    on="document_id", how="inner",
)
print(f"{len(_with_images):,} of {len(df):,} working-sample invoices have rendered images.")

N_SAMPLES_PER_CLASS = 6
_fraud = _with_images[_with_images["is_fraud"] == 1].sample(
    n=min(N_SAMPLES_PER_CLASS, (_with_images["is_fraud"] == 1).sum()),
    random_state=42,
)
_legit = _with_images[_with_images["is_fraud"] == 0].sample(
    n=min(N_SAMPLES_PER_CLASS, (_with_images["is_fraud"] == 0).sum()),
    random_state=42,
)
real_df = (pd.concat([_fraud, _legit], ignore_index=True)
             .sample(frac=1, random_state=42)
             .reset_index(drop=True))
real_df["image_filename"] = real_df["document_id"] + ".png"

# Score with the trained RF. `real_df` already has every enrichment
# feature (supplier_risk_score, blacklisted_flag, split_invoice_flag,
# late_night_flag, amount_zscore) so this is a straight re-featurise.
_rf_in = real_df.copy()
_rf_in["doc_type_encoded"] = _rf_in["document_type"].astype("category").cat.codes
_rf_in["amount_log"]       = np.log1p(_rf_in["amount"])
_rf_in["amount_gt_250"]    = (_rf_in["amount"] > 250).astype(int)
_rf_in["day_of_week"]      = pd.to_datetime(_rf_in["date_created"]).dt.dayofweek
_rf_in["vendor_frequency"] = _rf_in["vendor_name"].map(
    ml_results["vendor_counts"]
).fillna(1).astype(int)
real_df["fraud_probability"] = ml_results["model"].predict_proba(
    _rf_in[ml_results["feature_cols"]]
)[:, 1]


def run_real_pipeline(real_df, images_dir=IMAGES_DIR, top_k_agents=3, sleep_s=7.0,
                      use_dual_vision: bool = False):
    """Full image + metadata + multi-agent on every real document.
    Rate-limited (~8.5 RPM) to stay under the Gemini Flash free-tier 10 RPM cap.

    Vision dispatch: respects the cell-1 `USE_NEMOTRON_VISION` toggle through
    `vision_analyze`. Pass `use_dual_vision=True` to override and run both
    Gemma + Nemotron in parallel via `load_and_analyze_image_dual`."""
    top_k_ids = set(real_df.nlargest(top_k_agents, 'fraud_probability')['document_id'])
    vision_fn = (load_and_analyze_image_dual
                 if (use_dual_vision and _openrouter is not None)
                 else vision_analyze)
    rows = []
    for _, row in real_df.iterrows():
        doc_id = row['document_id']
        path = os.path.join(images_dir, row['image_filename'])
        print(f"[{doc_id}] vision ({vision_fn.__name__}): {row['image_filename']} ...", flush=True)
        try:
            image_analysis = vision_fn(path)
        except Exception as exc:
            image_analysis = {'error': str(exc), 'fraud_risk_score': 0}
        time.sleep(sleep_s)

        metadata_pred = {'fraud_probability': float(row['fraud_probability'])}

        agent_assessment = None
        if doc_id in top_k_ids:
            print(f"[{doc_id}] multi-agent assessment ...", flush=True)
            try:
                agent_assessment = multi_agent_assessment(row.to_dict(), sleep_s=sleep_s)
            except Exception as exc:
                agent_assessment = {'error': str(exc), 'avg_risk_score': 0, 'opinions': [], 'consensus': 'error'}

        integrated = integrate_all_analyses(doc_id, image_analysis, metadata_pred, agent_assessment)
        integrated['vendor_name']     = row['vendor_name']
        integrated['amount']          = row['amount']
        integrated['image_filename']  = row['image_filename']
        integrated['is_fraud_label']  = int(row['is_fraud'])
        integrated['fraud_type']      = row.get('fraud_type', 'NONE')
        integrated['vision_summary']  = (image_analysis or {}).get('summary', '')
        rows.append(integrated)
    return pd.DataFrame(rows)


print("=" * 64)
print(f"REAL DOCUMENT MULTIMODAL PIPELINE — {len(real_df)} Kaggle sample images")
print("=" * 64)
print(f"Class balance: {int(real_df['is_fraud'].sum())} fraud / "
      f"{int((real_df['is_fraud'] == 0).sum())} legit")
real_results = run_real_pipeline(real_df)

print("\n" + "=" * 64)
print("RESULTS TABLE")
print("=" * 64)
print(real_results[['document_id', 'image_filename', 'is_fraud_label', 'fraud_type',
                    'amount', 'composite_risk_score', 'recommendation']].to_string(index=False))

print("\n" + "=" * 64)
print("HIGHEST-RISK REPORT")
print("=" * 64)
top_real = real_results.nlargest(1, 'composite_risk_score').iloc[0].to_dict()
print(generate_risk_report(top_real['document_id'], top_real))

real_results.to_csv('real_document_results.csv', index=False)
print(f"\nSaved {len(real_results)} real-document results to real_document_results.csv")


## 7. RL Optimization Analysis

We do **not** implement an RL agent — we analyze how RL concepts could optimize a fraud detection pipeline.

In [ ]:
def analyze_rl_optimization_potential() -> str:
    analysis = """
# Reinforcement Learning Optimization Analysis for Fraud Detection

## 1. Framing Fraud Detection as a Markov Decision Process (MDP)

The first thing we need to do is map our pipeline onto the MDP framework — the
formal backbone of any RL system. An MDP is defined by the tuple (S, A, P, R, γ):

- **States S**: each state represents a document arriving for review, encoded as
  its feature vector (amount, vendor, weekend flag, RF fraud probability, etc.).
  In practice the agent never sees the *full* truth — it works from metadata,
  image analysis scores, and expert opinions, which are partial and noisy. This
  makes the problem closer to a **POMDP** (Partially Observable MDP): the true
  fraudulence of a document is hidden, and our agent only receives observations
  (the features and model scores) that approximate it — much like a trader who
  cannot see the entire market, or a robot whose sensors are incomplete.

- **Actions A**: the discrete routing decision the agent outputs — APPROVE,
  REVIEW, or REJECT. This is a small, finite action space, which means
  value-based methods like Q-learning or DQN are natural fits (as the lecture
  notes, DQN and PPO work well for discrete actions).

- **Transitions P(s'|s, a)**: the next state is simply the next document in the
  stream. Transitions here are largely independent of the action taken on the
  current document — unlike a grid-world where moving right changes your
  position, approving an invoice doesn't change which invoice comes next. The
  non-trivial part is that the *distribution* of incoming documents drifts over
  time as fraud patterns evolve.

- **Rewards R(s, a)**: this is where the design gets interesting. A naive +1/−1
  (correct/wrong) reward ignores the fact that fraud detection is deeply
  asymmetric — missing a $50,000 fraud is far worse than sending a legitimate
  $50 invoice to manual review. A more realistic reward function would be:

      R = +C_tp          if we correctly catch a fraud
          −C_fn × amount if we miss a fraud (scaled by how much money is lost)
          −C_fp × review_cost  if we flag a legitimate document (wasted analyst time)

  This kind of shaped reward teaches the agent that not all mistakes are equal,
  and that high-value frauds deserve more aggressive flagging.

- **Discount factor γ**: fraud confirmations often arrive weeks later (chargebacks,
  audit results). A γ close to 1 (say 0.95–0.99) tells the agent to care about
  these delayed outcomes — to be patient and value long-term accuracy over
  short-term convenience, much like setting γ = 0.99 for a far-sighted agent
  versus γ = 0.5 for a myopic one.

## 2. The Agent–Environment Loop

Following the standard RL loop covered in the lecture:

  1. The agent **observes** the current state (document features + model scores).
  2. It **chooses an action** (APPROVE / REVIEW / REJECT).
  3. The environment **returns a reward** (correct catch, missed fraud, or false alarm)
     and the agent **transitions** to the next document.
  4. The agent **updates its policy** based on this feedback.

Over thousands of episodes (i.e., thousands of documents processed with eventual
ground-truth labels), the agent would gradually learn which combinations of
features and scores should trigger escalation — just like the FrozenLake agent
that starts by falling into holes randomly but eventually discovers a safe path
through trial and error.

## 3. Exploration vs. Exploitation

This is arguably the most critical RL concept for fraud detection. A purely
greedy policy — one that always exploits what it currently knows — will lock
onto familiar fraud patterns and become blind to novel schemes. Fraudsters
adapt, and a system that only exploits will be outmaneuvered.

- **ε-greedy exploration**: with probability ε, the agent takes a random action
  (e.g., sends a low-risk document to manual review anyway). This generates
  ground-truth labels for documents the agent would otherwise auto-approve,
  keeping its knowledge fresh. Over time, ε decays — early on the system
  explores aggressively to learn, then gradually shifts to exploiting what it
  has learned, exactly like the decaying ε schedule from the lecture.

- **Entropy-bonus exploration (SAC-style)**: the lecture introduced Soft
  Actor-Critic, which adds an entropy term to the reward to *encourage*
  exploration and prevent the policy from collapsing onto a single action too
  early. In our context, this would mean the agent maintains some uncertainty
  in its routing decisions, which is actually desirable — we *want* it to
  occasionally send borderline documents to review rather than always
  rubber-stamping them.

Without deliberate exploration, the system develops blind spots that fraud
rings can exploit. This is not a theoretical concern — it is the real-world
version of the FrozenLake agent that never discovers the path to the goal
because it keeps repeating the same moves.

## 4. Learning Methods — What Would Work Here

The lecture presented a progression of RL methods, and several would apply:

- **Q-learning**: since our action space is small (three actions), we could
  maintain a Q-table where each entry Q(s, a) estimates the expected
  cumulative reward of taking action a in state s. The Q-learning update rule
  — Q(s,a) ← Q(s,a) + α[r + γ max Q(s',a') − Q(s,a)] — would let the
  agent learn from each document's eventual outcome without needing a model
  of the environment. The "surprise" term (target minus old estimate) drives
  learning: if a REVIEW decision turns out better than expected, Q increases;
  if worse, it decreases.

- **Deep Q-Network (DQN)**: in practice, the state space is too large for a
  lookup table (13+ continuous features means effectively infinite states).
  A DQN replaces the Q-table with a neural network that takes the feature
  vector as input and outputs Q-values for each action. The network learns
  to *generalize* — instead of memorizing every document individually, it
  learns patterns like "high amount + weekend + duplicate flag → high Q for
  REJECT." Experience replay (storing past transitions and sampling random
  mini-batches) would be especially useful here, since fraud cases are rare
  and we want the network to revisit them often rather than forgetting them
  in the stream of legitimate documents.

- **PPO (Proximal Policy Optimization)**: the lecture's recommendation was
  "when in doubt, use PPO." For our pipeline, PPO would learn the routing
  policy π(a|s) directly — a neural network that outputs the probability of
  each action given the document features. PPO's clipped updates prevent the
  policy from changing too drastically in a single step, which is important
  in a domain where stability matters (we cannot afford the system suddenly
  flagging everything or approving everything after one bad batch of updates).

## 5. Concrete Improvements to Our Pipeline

Here is where RL concepts connect directly to the code we have built:

- **Adaptive thresholds**: the fixed REVIEW ≥ 35 and REJECT ≥ 65 thresholds
  in `integrate_all_analyses` are hand-tuned constants. An RL agent could
  learn to adjust these per document type or per risk profile, treating the
  threshold selection as actions whose Q-values are updated based on
  confirmed outcomes. The Bellman equation ensures these values propagate
  backward — a threshold that catches fraud today increases the value of
  the state that led to choosing it.

- **Smart expert routing**: `process_document_batch` currently runs all three
  expert agents on the top-k highest-risk documents. Each LLM call costs
  time and money. An RL agent could learn *which* expert to consult for each
  document — maybe the forensic accountant is best for high-amount invoices
  while the compliance officer adds more value for duplicate-vendor cases.
  This is a classic discrete-action RL problem: choose 1-of-3 experts (or
  none), observe whether the final classification improves, and update Q
  accordingly. This could cut API costs by ~3x while maintaining accuracy.

- **Active learning under uncertainty**: documents where the RF probability
  hovers around 0.4–0.6 are the ones the model is least sure about. An RL
  agent could learn to route these specifically to human review — not because
  the composite score is high, but because the *information gain* from
  labeling them is highest. This is exploration with a purpose: we spend
  review effort where it teaches the system the most.

## 6. Limitations and Honest Caveats

RL is not a silver bullet for fraud detection, and it is important to be
realistic:

- **Sparse, delayed rewards**: unlike Atari where the score updates every
  frame, fraud labels may take weeks to arrive (chargebacks, audit cycles).
  This makes credit assignment hard — the agent needs to connect today's
  routing decision to a reward signal that arrives much later. A high γ
  helps, but the practical challenge of collecting timely labels remains.

- **Adversarial environment**: the lecture examples (FrozenLake, Atari, Go)
  involve environments that do not deliberately fight back. Fraudsters do.
  They probe the system, find its blind spots, and adapt. This means the
  environment is non-stationary in a way that standard RL assumptions do
  not fully cover — the transition probabilities P(s'|s,a) shift as
  adversaries react to the policy.

- **Regulatory constraints**: in financial services, fully automated REJECT
  decisions often face legal scrutiny. RL should optimize the *triage*
  (which documents go to human review, and in what priority order), not
  the final adjudication. The human stays in the loop — RL just makes
  their queue smarter.

- **Sample efficiency**: fraud is rare (~22% in our dataset, much less in
  real life). The agent needs many episodes to learn, but each "episode"
  involves real money and real risk. Techniques like experience replay
  (reusing past transitions) help, but the cold-start problem is real —
  you need a decent supervised baseline (like our RF) before RL can
  meaningfully improve on it.
""".strip()
    return analysis


print(analyze_rl_optimization_potential())

## 8. Export Results

In [ ]:
def export_results(results, filename: str) -> str:
    timestamp = datetime.utcnow().isoformat()
    if isinstance(results, pd.DataFrame):
        payload = results.to_dict(orient='records')
    else:
        payload = results
    wrapped = {'generated_at': timestamp, 'n_records': len(payload), 'results': payload}

    if filename.endswith('.json'):
        with open(filename, 'w') as f:
            json.dump(wrapped, f, indent=2, default=str)
    elif filename.endswith('.csv') and isinstance(results, pd.DataFrame):
        results.to_csv(filename, index=False)
    else:
        raise ValueError("Use .json or .csv (DataFrame only for CSV)")
    return f"Exported {len(payload)} records to {filename}"


print(export_results(batch_results, 'fraud_batch_results.csv'))
print(export_results(batch_results, 'fraud_batch_results.json'))

# Example single-document report
top = batch_results.nlargest(1, 'composite_risk_score').iloc[0].to_dict()
print("\n" + generate_risk_report(top['document_id'], top))

## 9. Automated Email Alerts for High-Risk Documents

`EmailAlerter` sends a Gmail SMTP notification whenever a scored document crosses the alert threshold (default: `REJECT` recommendation OR composite score ≥ 65).

**Setup** — create a Google App Password (requires 2FA on the Gmail account) at myaccount.google.com/apppasswords, then export:

```bash
export GMAIL_USER="youraccount@gmail.com"
export GMAIL_APP_PASSWORD=""   # add a local App Password; never commit it
export FRAUD_ALERT_RECIPIENT="fraud-ops@yourcompany.com"
```

**Transport**: `smtp.gmail.com:587` with STARTTLS. A single persistent connection is reused across a batch, and an in-memory set of already-alerted `document_id`s prevents duplicate sends when the pipeline is re-run.

In [ ]:
import smtplib
import ssl
from email.message import EmailMessage
from email.utils import make_msgid, formatdate

GMAIL_SMTP_HOST = "smtp.gmail.com"
GMAIL_SMTP_PORT = 587
ALERT_SCORE_THRESHOLD = 65   # composite score at/above this triggers an alert
ALERT_RECOMMENDATIONS = {"REJECT"}  # recommendations that trigger regardless of score


class EmailAlerter:
    """Gmail SMTP alerter for high-risk fraud documents.

    Credentials are read from env (GMAIL_USER, GMAIL_APP_PASSWORD). The
    alerter keeps an in-memory set of already-notified document IDs so the
    same alert isn't re-sent if the pipeline runs twice in the same session.

    Usage:
        alerter = EmailAlerter(recipient="fraud-ops@company.com")
        alerter.alert_batch(batch_results)
    """

    def __init__(self, recipient: str | None = None, sender: str | None = None,
                 app_password: str | None = None, dry_run: bool | None = None):
        self.sender = sender or os.environ.get("GMAIL_USER")
        self.app_password = app_password or os.environ.get("GMAIL_APP_PASSWORD")
        self.recipient = recipient or os.environ.get("FRAUD_ALERT_RECIPIENT") or self.sender
        # Auto dry-run when credentials aren't configured — lets the notebook
        # demo the flow without leaking email to a real inbox.
        self.dry_run = dry_run if dry_run is not None else not (self.sender and self.app_password)
        self._sent_ids: set[str] = set()

    def _build_message(self, doc: dict) -> EmailMessage:
        doc_id = doc["document_id"]
        score = doc["composite_risk_score"]
        rec = doc["recommendation"]

        msg = EmailMessage()
        msg["Subject"] = f"[FRAUD ALERT] {rec} — {doc_id} (risk {score}/100)"
        msg["From"] = self.sender or "fraud-detector@localhost"
        msg["To"] = self.recipient
        msg["Date"] = formatdate(localtime=True)
        msg["Message-ID"] = make_msgid(domain="fraud-detector.local")
        msg["X-Fraud-Doc-ID"] = doc_id
        msg["X-Fraud-Score"] = str(score)

        report = generate_risk_report(doc_id, doc)
        body = (
            f"Automated fraud alert generated at {datetime.utcnow().isoformat()}Z\n\n"
            f"{report}\n\n"
            f"-- \nThis alert was sent automatically by the fraud-detection pipeline.\n"
            f"Do not reply; route investigations through the normal casebook system."
        )
        msg.set_content(body)
        return msg

    def _open_connection(self) -> smtplib.SMTP:
        context = ssl.create_default_context()
        server = smtplib.SMTP(GMAIL_SMTP_HOST, GMAIL_SMTP_PORT, timeout=30)
        server.ehlo()
        server.starttls(context=context)
        server.ehlo()
        server.login(self.sender, self.app_password)
        return server

    def alert_batch(self, batch: pd.DataFrame,
                    score_threshold: float = ALERT_SCORE_THRESHOLD,
                    recommendations: set[str] = ALERT_RECOMMENDATIONS) -> pd.DataFrame:
        """Send one email per high-risk document. Returns the subset that was alerted."""
        high_risk = batch[
            (batch["composite_risk_score"] >= score_threshold)
            | (batch["recommendation"].isin(recommendations))
        ]
        to_send = high_risk[~high_risk["document_id"].isin(self._sent_ids)]

        if to_send.empty:
            print("No new high-risk documents to alert on.")
            return to_send

        print(f"Preparing {len(to_send)} fraud alert(s) "
              f"(dry_run={self.dry_run}, recipient={self.recipient})")

        if self.dry_run:
            for _, row in to_send.iterrows():
                msg = self._build_message(row.to_dict())
                print(f"  [DRY-RUN] would send: {msg['Subject']}")
                self._sent_ids.add(row["document_id"])
            return to_send

        server = self._open_connection()
        try:
            for _, row in to_send.iterrows():
                msg = self._build_message(row.to_dict())
                try:
                    server.send_message(msg)
                    self._sent_ids.add(row["document_id"])
                    print(f"  sent: {msg['Subject']}")
                except smtplib.SMTPException as exc:
                    print(f"  FAILED {row['document_id']}: {exc}")
        finally:
            server.quit()
        return to_send


# Demo — runs in dry-run mode unless GMAIL_USER / GMAIL_APP_PASSWORD are set.
#alerter = EmailAlerter()
#alerted = alerter.alert_batch(batch_results)
#print(f"\nAlerted on {len(alerted)} document(s). Already-alerted cache size: {len(alerter._sent_ids)}")
#print("Re-running alert_batch is idempotent (no duplicate sends):")
#alerter.alert_batch(batch_results)

## How to run the full pipeline with live API calls

1. Make sure `GEMINI_API_KEY` is set (in the environment (for example via a local .env file)).
2. Run cells 1-3 (setup + data).
3. Run cell 5 (`analyze_metadata_ml`) — fills `ml_results`.
4. **Optional (API)**: uncomment the call in cell 6 to run `analyze_metadata_llm`.
5. **Optional (API)**: uncomment the call in cell 7 to run `multi_agent_assessment` on a sample doc.
6. Cell 8 builds `batch_results`. To run full pipeline with agents on top-k:
   `batch_results = process_document_batch(df, ml_results, image_folder_path=None, run_agents_for_top_k=3)`
7. Cell 9 renders the dashboard.
8. Cell 10 prints the RL analysis.
9. Cell 11 exports CSV + JSON.